### Import Libraries

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

### File path and model name configuration
Change MODEL_NAME:
  - MDeBERTA: "microsoft/mdeberta-v3-base"
  - XLMR: "xlm-roberta-base"

In [ ]:
MODEL_NAME = "microsoft/mdeberta-v3-base"
Lang="SW"
TEXT_COLUMN = "sentence"
LABEL_COLUMN = "label"
INPUT_DIR=''
chunked=True
OUTPUT_DIR = (
    f"mdeberta-finetuned-translator-detector-{Lang}-{MODEL_NAME[:9]}-chunk"
    if chunked else
    f"mdeberta-finetuned-translator-detector-{Lang}-{MODEL_NAME}"
)

MAX_CHUNK_TOKENS = 300

### Data Loading and Cleaning

In [ ]:
import re

def clean_text(s):
    if isinstance(s, str):
        s = s.strip()                         # remove leading/trailing spaces
        s = re.sub(r"\s+", " ", s)            # collapse multiple whitespace into one space
    return s



In [ ]:
# Load tokenizer first, since chunking depends on token counts
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

In [ ]:
def count_tokens(text, tokenizer):
    """
    Count tokens without adding special tokens.
    """
    return len(tokenizer.encode(text, add_special_tokens=False))

In [ ]:
def chunk_dataframe_by_label(
    df: pd.DataFrame,
    tokenizer,
    text_col: str = "sentence",
    label_col: str = "label",
    max_tokens: int = 300
) -> pd.DataFrame:
    """
    For each label group:
      - keep sentence order as it appears in the file
      - concatenate sentences of the same label
      - each output row contains up to max_tokens tokens
    """
    chunked_rows = []

    # Make sure the dataframe only has clean valid rows
    df = df[[text_col, label_col]].copy()
    df = df.dropna(subset=[text_col, label_col])
    df[text_col] = df[text_col].astype(str)
    df[label_col] = df[label_col].astype(int)

    for label_value in sorted(df[label_col].unique()):
        group_df = df[df[label_col] == label_value]

        current_sentences = []
        current_token_count = 0

        for sentence in group_df[text_col]:
            sentence = sentence.strip()
            if not sentence:
                continue

            sent_tokens = count_tokens(sentence, tokenizer)

            # If a single sentence is longer than max_tokens,
            # keep it as its own row, truncated at token boundary if needed
            if sent_tokens > max_tokens:
                # flush current chunk first
                if current_sentences:
                    chunked_rows.append({
                        text_col: " ".join(current_sentences),
                        label_col: label_value
                    })
                    current_sentences = []
                    current_token_count = 0

                token_ids = tokenizer.encode(sentence, add_special_tokens=False)[:max_tokens]
                truncated_text = tokenizer.decode(token_ids, skip_special_tokens=True)

                chunked_rows.append({
                    text_col: truncated_text,
                    label_col: label_value
                })
                continue

            # If adding this sentence exceeds the limit, flush current chunk
            if current_token_count + sent_tokens > max_tokens:
                if current_sentences:
                    chunked_rows.append({
                        text_col: " ".join(current_sentences),
                        label_col: label_value
                    })

                current_sentences = [sentence]
                current_token_count = sent_tokens
            else:
                current_sentences.append(sentence)
                current_token_count += sent_tokens

        # flush remaining chunk for this label
        if current_sentences:
            chunked_rows.append({
                text_col: " ".join(current_sentences),
                label_col: label_value
            })

    return pd.DataFrame(chunked_rows)

In [ ]:
# Load CSV files
try:
    train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
    val_df = pd.read_csv(os.path.join(INPUT_DIR, "val.csv"))
except FileNotFoundError as e:
    print(f"Error: {e}. Make sure train.csv and val.csv are in the directory.")
    raise

In [ ]:
# Clean text
for df in [train_df, val_df]:
    df[TEXT_COLUMN] = df[TEXT_COLUMN].apply(clean_text)

print("Before chunking:")
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
train_df.head()

In [ ]:
if chunked:
    train_df = chunk_dataframe_by_label(
        train_df,
        tokenizer=tokenizer,
        text_col=TEXT_COLUMN,
        label_col=LABEL_COLUMN,
        max_tokens=MAX_CHUNK_TOKENS
    )

    val_df = chunk_dataframe_by_label(
        val_df,
        tokenizer=tokenizer,
        text_col=TEXT_COLUMN,
        label_col=LABEL_COLUMN,
        max_tokens=MAX_CHUNK_TOKENS
    )
    # shuffle after chunking
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
print("After chunking:")
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
train_df.head()

In [ ]:
# Save chunked versions for inspection
if chunked:
    train_chunked_path = os.path.join(INPUT_DIR, "train_chunked_300.csv")
    val_chunked_path = os.path.join(INPUT_DIR, "val_chunked_300.csv")

    train_df.to_csv(train_chunked_path, index=False)
    val_df.to_csv(val_chunked_path, index=False)

    print(f"Saved chunked train file to: {train_chunked_path}")
    print(f"Saved chunked val file to: {val_chunked_path}")

In [ ]:
# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

raw_datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print("Datasets loaded successfully:")
print(raw_datasets)
print(raw_datasets["train"][0])

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding="max_length",
        truncation=True
    )

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COLUMN])
tokenized_datasets = tokenized_datasets.rename_column(LABEL_COLUMN, "labels")
tokenized_datasets.set_format("torch")

print("\nTokenization complete. Sample from training set:")
print(tokenized_datasets["train"][0])

### Load Model

In [ ]:
# Load the pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

In [ ]:
# This function will be called during evaluation to compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Get the predictions by taking the argmax of the logits
    predictions = logits.argmax(axis=-1)
    
    # Calculate precision, recall, and F1-score
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )
    # Calculate accuracy
    acc = accuracy_score(labels, predictions)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

#### Model Configuration

In [ ]:
if chunked:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=6,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        learning_rate=1e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=True,
        report_to="none",
        seed=42,
    )
else:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        learning_rate=1.5e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=True,
        report_to="none",
        seed=42,
    )

### Train and Save Model

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Start fine-tuning the model
print("\n Starting model fine-tuning...")
trainer.train()
print(" Fine-tuning complete!")

In [ ]:
# Save the final model and tokenizer 
final_model_path = f"{OUTPUT_DIR}/final_model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"\n Final model and tokenizer saved to {final_model_path}")